# Affinity Classification Diagnostics

This notebook computes ordinal affinity diagnostics on `(abstract, question)` pairs.

Metrics included:
1. Per-model class metrics
2. Per-model quadratic weighted kappa
3. Per-model mean signed error
4. An additional synthetic model formed by averaging predictions across models

In [1]:
from pathlib import Path

import importlib
import itertools
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import GroupKFold
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [2]:
# Label definitions (edit only if your taxonomy changes).
LABEL_ORDER = ["Low", "Moderate", "High"]
LABEL_TO_NUM = {"Low": 0, "Moderate": 1, "High": 2}
NUM_TO_LABEL = {0: "Low", 1: "Moderate", 2: "High"}


def normalize_label(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, np.integer, float, np.floating)):
        x = float(v)
        # Numeric predictions in merged files are raw affinity scores (0-120).
        if 0 <= x < 40:
            return "Low"
        if 40 <= x < 80:
            return "Moderate"
        if x >= 80:
            return "High"
        return np.nan
    text = str(v).strip()
    text_low = text.lower()
    if text_low in {"low", "moderate", "high"}:
        return text_low.capitalize()
    return np.nan


# Hardcoded column names used throughout this notebook.
expert_col = "Evaluator_Affinity_Level"
prediction_col = "LLM_Affinity"
abstract_id_col = "Abstract_Index"
model_col = "Model_Slug"

## Load Data

The loader tries likely project files first (including benchmark and calibration outputs), then you can override `input_path` manually if needed.

In [3]:
repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bench_root = repo_root / "data" / "results" / "affinity_benchmark"
calib_root = repo_root / "data" / "processed" / "affinity_calibration"
benchmark_input_filename = "merged_ra_affinities.csv"
calibration_input_filename = "calibration_abstract_question_affinities.csv"

# benchmark_run_dir_names = ["20260422_212121_1", "20260422_212121_4", "20260422_212121_3"]
benchmark_run_dir_names = ["20260726_134607_5"]
# benchmark_run_dir_names = ["20260726_154151_3"]
# benchmark_run_dir_names = ["20260726_134607_1"]
    
calibration_run_dir_names = ["20260728_abstracts_evaluation_template_mark_om_v1"]

def load_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin1"):
            try:
                return pd.read_csv(path, encoding=encoding)
            except UnicodeDecodeError:
                continue
        raise UnicodeDecodeError(
            "utf-8", b"", 0, 1, f"Unable to decode CSV file: {path}"
        )
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file extension for {path}")

def collect_frames(root: Path, run_dir_names, input_filename: str, source_name: str):
    if isinstance(run_dir_names, (str, Path)):
        run_dir_names = [run_dir_names]

    run_dirs = sorted([p for p in root.iterdir() if p.is_dir()])
    if not run_dirs:
        raise FileNotFoundError(f"No run directories found under {root}")

    selected_run_dirs = [p for p in run_dirs if p.name in {str(x) for x in run_dir_names}]
    if not selected_run_dirs:
        raise FileNotFoundError(f"No matching run directories found for: {run_dir_names}")

    frames = []
    for run_dir in selected_run_dirs:
        input_path = run_dir / input_filename
        if not input_path.exists():
            raise FileNotFoundError(f"Missing file: {input_path}")

        run_df = load_table(input_path)
        run_df["Run_ID"] = run_dir.name
        run_df["Source_Type"] = source_name
        frames.append(run_df)

    return frames, selected_run_dirs

benchmark_frames, selected_benchmark_dirs = collect_frames(
    bench_root, benchmark_run_dir_names, benchmark_input_filename, "benchmark"
)
calibration_frames, selected_calibration_dirs = collect_frames(
    calib_root, calibration_run_dir_names, calibration_input_filename, "calibration"
)

bench_df = pd.concat(benchmark_frames, ignore_index=True)
calib_df = pd.concat(calibration_frames, ignore_index=True)
df_raw = pd.merge(
    bench_df,
    calib_df,
    on=["Abstract_Index", "RA2025_ID"],
    how="inner",
    suffixes=("_bench", "_calib"),
)

print("Benchmark run directories:", [p.name for p in selected_benchmark_dirs])
print("Calibration run directories:", [p.name for p in selected_calibration_dirs])
print("Calibration abstracts:", calib_df['Abstract_Index'].nunique())
print("Benchmark abstracts:", bench_df['Abstract_Index'].nunique())
print("Rows loaded:", len(df_raw))
print(f"Columns ({len(df_raw.columns)}): {list(df_raw.columns)}")
display(df_raw.head(10))

Benchmark run directories: ['20260726_134607_5']
Calibration run directories: ['20260728_abstracts_evaluation_template_mark_om_v1']
Calibration abstracts: 91
Benchmark abstracts: 105
Rows loaded: 3010
Columns (22): ['Abstract_Index', 'Document Title_bench', 'RA2025_ID', 'RA_Question_bench', 'Cosine_x100', 'LLM_Affinity', 'LLM_Affinity_Reason', 'Was_Repaired', 'Repair_Method', 'Repair_Attempt', 'Repaired_At_UTC', 'Model_Slug', 'Run_ID_bench', 'Source_Type_bench', 'Document Title_calib', 'RA_Question_calib', 'Evaluator_Affinity', 'Evaluator_Affinity_Level', 'Evaluator_Affinity_Reason', 'Source_Evaluator', 'Run_ID_calib', 'Source_Type_calib']


,Abstract_Index,Document Title_bench,RA2025_ID,RA_Question_bench,Cosine_x100,LLM_Affinity,LLM_Affinity_Reason,Was_Repaired,Repair_Method,Repair_Attempt,...,Run_ID_bench,Source_Type_bench,Document Title_calib,RA_Question_calib,Evaluator_Affinity,Evaluator_Affinity_Level,Evaluator_Affinity_Reason,Source_Evaluator,Run_ID_calib,Source_Type_calib
0,10.1109/TPWRS.2023.3292799,Optimal Primary Frequency Reserve Provision by...,35,"what models and methods, including testing, ar...",49.4778,35.0,Abstract addresses reliability services (frequ...,False,NaN,NaN,...,20260726_134607_5,benchmark,Optimal Primary Frequency Reserve Provision by...,"What models and methods, including testing, ar...",70,Moderate,NaN,Abstracts Evaluation Template - Mark OM.xlsm,20260728_abstracts_evaluation_template_mark_om_v1,calibration
1,10.1109/TPWRS.2023.3292799,Optimal Primary Frequency Reserve Provision by...,18,what are the relative merits of different coor...,40.4109,80.0,Abstract directly addresses DER coordination a...,False,NaN,NaN,...,20260726_134607_5,benchmark,Optimal Primary Frequency Reserve Provision by...,What are the relative merits of different coor...,10,Low,NaN,Abstracts Evaluation Template - Mark OM.xlsm,20260728_abstracts_evaluation_template_mark_om_v1,calibration
2,10.1109/TPWRS.2023.3292799,Optimal Primary Frequency Reserve Provision by...,12,what mechanisms and appropriate aggregate der ...,39.5109,85.0,Abstract presents aggregate DER representation...,False,NaN,NaN,...,20260726_134607_5,benchmark,Optimal Primary Frequency Reserve Provision by...,What mechanisms and appropriate aggregate DER ...,50,Moderate,NaN,Abstracts Evaluation Template - Mark OM.xlsm,20260728_abstracts_evaluation_template_mark_om_v1,calibration
3,10.1109/TPWRS.2023.3292799,Optimal Primary Frequency Reserve Provision by...,15,"what aggregate quantities must be monitored, s...",36.7361,60.0,Abstract addresses monitoring and validation o...,False,NaN,NaN,...,20260726_134607_5,benchmark,Optimal Primary Frequency Reserve Provision by...,"What aggregate quantities must be monitored, s...",10,Low,NaN,Abstracts Evaluation Template - Mark OM.xlsm,20260728_abstracts_evaluation_template_mark_om_v1,calibration
4,10.1109/TPWRS.2023.3292799,Optimal Primary Frequency Reserve Provision by...,17,what is a scalable data architecture for der m...,20.6269,55.0,Abstract addresses DER monitoring and coordina...,False,NaN,NaN,...,20260726_134607_5,benchmark,Optimal Primary Frequency Reserve Provision by...,What is a scalable data architecture for DER m...,10,Low,NaN,Abstracts Evaluation Template - Mark OM.xlsm,20260728_abstracts_evaluation_template_mark_om_v1,calibration
5,10.1109/TPWRS.2023.3292799,Optimal Primary Frequency Reserve Provision by...,16,what is the communication capability needed to...,18.2876,35.0,Abstract implies communication needs for aggre...,False,NaN,NaN,...,20260726_134607_5,benchmark,Optimal Primary Frequency Reserve Provision by...,What is the communication capability needed to...,10,Low,NaN,Abstracts Evaluation Template - Mark OM.xlsm,20260728_abstracts_evaluation_template_mark_om_v1,calibration
6,10.1109/TPWRS.2023.3242715,Graph Learning-Based Voltage Regulation in Dis...,13,how can system operators quantify the bulk-lev...,41.3375,25.0,Abstract addresses voltage regulation services...,False,NaN,NaN,...,20260726_134607_5,benchmark,Graph Learning-Based Voltage Regulation in Dis...,How can system operators quantify the bulk-lev...,50,Moderate,NaN,Abstracts Evaluation Template - Mark OM.xlsm,20260728_abstracts_evaluation_template_mark_om_v1,calibration
7,10.1109/TPWRS.2023.3242715,Graph Learning-Based Voltage Regulation in Dis...,18,what are the relative merits of different coor...,38.0265,55.0,Proposes decentralized coordination architectu...,False,NaN,NaN,...,20260726_134607_5,benchmark,Graph Learning-Based Voltage Regulation in Dis...,What are the relative merits of different coor...,10,Low,NaN,Abstracts Evaluation Template - Mark OM.xlsm,20260728_abstracts_eval

In [4]:
required_cols = [expert_col, prediction_col, abstract_id_col]
missing_required = [c for c in required_cols if c not in df_raw.columns]

print("Hardcoded column mapping in use:")
print(f"  expert_col      -> {expert_col}")
print(f"  prediction_col  -> {prediction_col}")
print(f"  abstract_id_col -> {abstract_id_col}")

if missing_required:
    raise ValueError(
        f"Missing required columns in df_raw: {missing_required}. "
        "Please update hardcoded column names near the top of this notebook."
    )

Hardcoded column mapping in use:
  expert_col      -> Evaluator_Affinity_Level
  prediction_col  -> LLM_Affinity
  abstract_id_col -> Abstract_Index


## Clean Labels

The cleaner normalizes variants such as `low`, `LOW`, `med`, `medium`, and `high`, then maps labels to ordinal values:

- Low = 0
- Medium = 1
- High = 2

In [5]:
# Assume labels are canonical: 'Low', 'Moderate', 'High'.
eval_df = df_raw.copy()
rows_before = len(eval_df)

expert_labels = eval_df[expert_col].map(normalize_label)
pred_labels = eval_df[prediction_col].map(normalize_label)
valid_mask = expert_labels.notna() & pred_labels.notna()

eval_df = eval_df.loc[valid_mask].copy()
eval_df['expert_num'] = expert_labels.loc[valid_mask].map(LABEL_TO_NUM).astype(int).to_numpy()
eval_df['pred_num'] = pred_labels.loc[valid_mask].map(LABEL_TO_NUM).astype(int).to_numpy()

rows_after = len(eval_df)
rows_removed = rows_before - rows_after

print(f'Rows before cleaning: {rows_before:,}')
print(f'Rows after cleaning:  {rows_after:,}')
print(f'Rows removed:         {rows_removed:,}')

print(f'Evaluated (abstract, question) pairs after cleaning: {rows_after:,}')
if abstract_id_col in eval_df.columns:
    print(f'Unique abstracts: {eval_df[abstract_id_col].nunique():,}')
else:
    print('Unique abstracts: not available (abstract_id_col not found).')
print(f'Unique abstract indices: {df_raw.Abstract_Index.unique().size:,}')

Rows before cleaning: 3,010
Rows after cleaning:  3,010
Rows removed:         0
Evaluated (abstract, question) pairs after cleaning: 3,010
Unique abstracts: 91
Unique abstract indices: 91


## Model-wise Metrics

In [6]:
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

affinity_metrics = importlib.import_module("src.analysis.affinity_metrics")
AVERAGE_MODEL_NAME = "Average across models"


def build_average_model_frame(df: pd.DataFrame) -> pd.DataFrame:
    # Average raw 0-120 scores across models first, then discretise (matches the
    # project's Affinity_Mean convention), rather than averaging discrete levels.
    averaged_df = (
        df.groupby([abstract_id_col, "RA2025_ID"], as_index=False)
        .agg(
            expert_num=("expert_num", "first"),
            raw_score_mean=(prediction_col, "mean"),
        )
    )
    averaged_df["pred_num"] = (
        averaged_df["raw_score_mean"].map(normalize_label).map(LABEL_TO_NUM).astype(int)
    )
    return averaged_df


def build_metrics_for_model(df: pd.DataFrame, model_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    y_true = df["expert_num"].to_numpy()
    y_pred = df["pred_num"].to_numpy()

    class_metrics_df = affinity_metrics.build_affinity_class_metrics(
        y_true,
        y_pred,
        labels=list(LABEL_TO_NUM.values()),
        class_names=LABEL_ORDER,
    ).copy()
    class_metrics_df.insert(0, "Model", model_name)

    overall_metrics_df = pd.DataFrame(
        [
            {
                "Model": model_name,
                "Quadratic Weighted Kappa": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
                "Mean signed error": float(np.mean(y_pred - y_true)),
                "Pairs": int(len(df)),
                "Unique abstracts": int(df[abstract_id_col].nunique()),
            }
        ]
    )
    return class_metrics_df, overall_metrics_df

In [7]:
model_class_metrics_frames = []
model_overall_metrics_frames = []

for model_name, model_df in eval_df.groupby(model_col, sort=True):
    class_metrics_df, overall_metrics_df = build_metrics_for_model(model_df, model_name)
    model_class_metrics_frames.append(class_metrics_df)
    model_overall_metrics_frames.append(overall_metrics_df)

average_model_df = build_average_model_frame(eval_df)
average_class_metrics_df, average_overall_metrics_df = build_metrics_for_model(
    average_model_df,
    AVERAGE_MODEL_NAME,
)

class_metrics_df = pd.concat(
    model_class_metrics_frames + [average_class_metrics_df],
    ignore_index=True,
)
overall_metrics_df = pd.concat(
    model_overall_metrics_frames + [average_overall_metrics_df],
    ignore_index=True,
)

class_metrics_df["Class"] = pd.Categorical(class_metrics_df["Class"], categories=LABEL_ORDER, ordered=True)
class_metrics_df = class_metrics_df.sort_values(["Model", "Class"]).reset_index(drop=True)
overall_metrics_df = overall_metrics_df.sort_values("Model").reset_index(drop=True)

display(overall_metrics_df)
display(class_metrics_df[["Model", "Class", "Support", "Precision", "Recall", "One_vs_Rest_Accuracy"]])

,Model,Quadratic Weighted Kappa,Mean signed error,Pairs,Unique abstracts
0,Average across models,0.422301,0.181063,602,91
1,claude-haiku-4.5,0.385572,0.200997,602,91
2,deepseek-v4-flash-cloud,0.381463,0.119601,602,91
3,gemini-3.1-flash-lite,0.323367,0.533223,602,91
4,gemma4-31b-cloud,0.365192,0.392027,602,91
5,gpt-5.4-mini,0.354073,0.151163,602,91


,Model,Class,Support,Precision,Recall,One_vs_Rest_Accuracy
0,Average across models,Low,438,0.880597,0.673516,0.696013
1,Average across models,Moderate,137,0.358974,0.613139,0.662791
2,Average across models,High,27,0.303030,0.370370,0.933555
3,claude-haiku-4.5,Low,438,0.874214,0.634703,0.667774
4,claude-haiku-4.5,Moderate,137,0.316406,0.591241,0.616279
5,claude-haiku-4.5,High,27,0.214286,0.222222,0.928571
6,deepseek-v4-flash-cloud,Low,438,0.862069,0.742009,0.725914
7,deepseek-v4-flash-cloud,Moderate,137,0.411765,0.562044,0.717608
8,deepseek-v4-flash-cloud,High,27,0.210526,0.296296,0.918605
9,gemini-3.1-flash-lite,Low,438,0.930851,0.399543,0.541528


In [13]:
class_metrics_df[class_metrics_df["Class"] == "Moderate"][["Model", "Support", "Precision", "Recall", "One_vs_Rest_Accuracy"]]

,Model,Support,Precision,Recall,One_vs_Rest_Accuracy
1,Average across models,137,0.358974,0.613139,0.662791
4,claude-haiku-4.5,137,0.316406,0.591241,0.616279
7,deepseek-v4-flash-cloud,137,0.411765,0.562044,0.717608
10,gemini-3.1-flash-lite,137,0.234177,0.540146,0.493355
13,gemma4-31b-cloud,137,0.293286,0.605839,0.578073
16,gpt-5.4-mini,137,0.323077,0.306569,0.696013


In [8]:
# Compare each model against the "Average across models" baseline, per class.
# Support is excluded here: it's a sample count, not a higher-is-better metric,
# so "better/worse than average" doesn't apply to it.
delta_metrics = ["Precision", "Recall", "One_vs_Rest_Accuracy"]
colors = {"Low": "#636EFA", "Moderate": "#EF553B", "High": "#00CC96"}

average_metrics_df = (
    class_metrics_df.loc[class_metrics_df["Model"] == AVERAGE_MODEL_NAME, ["Class"] + delta_metrics]
    .rename(columns={metric: f"{metric}_avg" for metric in delta_metrics})
)

delta_df = class_metrics_df[class_metrics_df["Model"] != AVERAGE_MODEL_NAME].merge(
    average_metrics_df, on="Class", how="left"
)
for metric in delta_metrics:
    delta_df[f"{metric}_delta"] = delta_df[metric] - delta_df[f"{metric}_avg"]

model_order = sorted(delta_df["Model"].unique())

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"{m} (Δ vs average)" for m in delta_metrics],
    horizontal_spacing=0.08,
)

for col_idx, metric in enumerate(delta_metrics, start=1):
    for class_label in LABEL_ORDER:
        class_data = (
            delta_df[delta_df["Class"] == class_label]
            .set_index("Model")
            .reindex(model_order)
            .reset_index()
        )
        fig.add_trace(
            go.Bar(
                x=class_data["Model"],
                y=class_data[f"{metric}_delta"],
                name=class_label,
                marker=dict(color=colors[class_label], line=dict(color="#1f1f1f", width=0.5)),
                legendgroup=class_label,
                showlegend=(col_idx == 1),
                customdata=np.stack([class_data[metric], class_data[f"{metric}_avg"]], axis=-1),
                hovertemplate=(
                    f"<b>{class_label}</b><br>"
                    "Model: %{x}<br>"
                    f"{metric}: %{{customdata[0]:.3f}}<br>"
                    "Average: %{customdata[1]:.3f}<br>"
                    "Δ: %{y:+.3f}"
                    "<extra></extra>"
                ),
            ),
            row=1,
            col=col_idx,
        )
    fig.add_hline(
        y=0,
        line_width=1,
        line_color="#1f1f1f",
        annotation_text="Average",
        annotation_position="top left",
        row=1,
        col=col_idx,
    )
    fig.update_yaxes(title_text="Δ vs average", row=1, col=col_idx)
    fig.update_xaxes(title_text="Model", row=1, col=col_idx)

fig.update_layout(
    title_text="Model Performance Relative to Cross-Model Average",
    barmode="group",
    bargap=0.2,
    bargroupgap=0.05,
    height=500,
    hovermode="closest",
)

fig.show()


## Robustness of the Model Ranking

Quadratic weighted kappa is estimated from 601 pairs, so small gaps in the table above may be
noise. Resampling pairs with replacement gives an interval around each model's QWK and a paired
comparison between the ensemble and the strongest individual model.


In [9]:
BOOTSTRAP_DRAWS = 1000
BOOTSTRAP_SEED = 0

QWK_WEIGHTS = (np.subtract.outer(np.arange(3), np.arange(3)) ** 2) / 4.0


def quadratic_weighted_kappa(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Same value as cohen_kappa_score(..., weights="quadratic"), fast enough to bootstrap."""
    observed = np.zeros((3, 3))
    np.add.at(observed, (y_true, y_pred), 1)
    expected = np.outer(observed.sum(axis=1), observed.sum(axis=0)) / observed.sum()
    return 1.0 - (QWK_WEIGHTS * observed).sum() / (QWK_WEIGHTS * expected).sum()


pair_index = [abstract_id_col, "RA2025_ID"]
level_matrix = pd.DataFrame(
    {name: g.set_index(pair_index)["pred_num"] for name, g in eval_df.groupby(model_col, sort=True)}
).dropna().astype(int)
level_matrix[AVERAGE_MODEL_NAME] = average_model_df.set_index(pair_index)["pred_num"]
expert_num_by_pair = average_model_df.set_index(pair_index).loc[level_matrix.index, "expert_num"].to_numpy()

rng = np.random.default_rng(BOOTSTRAP_SEED)
draw_rows = rng.integers(0, len(level_matrix), size=(BOOTSTRAP_DRAWS, len(level_matrix)))
bootstrap_kappa = pd.DataFrame(
    {
        name: [
            quadratic_weighted_kappa(expert_num_by_pair[rows], level_matrix[name].to_numpy()[rows])
            for rows in draw_rows
        ]
        for name in level_matrix.columns
    }
)

qwk_ci_df = (
    pd.DataFrame(
        {
            "Model": level_matrix.columns,
            "QWK": [
                quadratic_weighted_kappa(expert_num_by_pair, level_matrix[name].to_numpy())
                for name in level_matrix.columns
            ],
            "CI lower (2.5%)": bootstrap_kappa.quantile(0.025).to_numpy(),
            "CI upper (97.5%)": bootstrap_kappa.quantile(0.975).to_numpy(),
        }
    )
    .sort_values("QWK", ascending=False)
    .reset_index(drop=True)
)

best_single_model = qwk_ci_df.loc[qwk_ci_df["Model"] != AVERAGE_MODEL_NAME, "Model"].iloc[0]
ensemble_gain = bootstrap_kappa[AVERAGE_MODEL_NAME] - bootstrap_kappa[best_single_model]

display(qwk_ci_df)
print(f"Ensemble minus best single model ({best_single_model}):")
print(f"  point estimate      {ensemble_gain.mean():+.3f}")
print(f"  95% interval        [{ensemble_gain.quantile(0.025):+.3f}, {ensemble_gain.quantile(0.975):+.3f}]")
print(f"  P(ensemble better)  {(ensemble_gain > 0).mean():.2f}")

,Model,QWK,CI lower (2.5%),CI upper (97.5%)
0,Average across models,0.422301,0.343295,0.499448
1,claude-haiku-4.5,0.385572,0.319880,0.453636
2,deepseek-v4-flash-cloud,0.381463,0.308250,0.463779
3,gemma4-31b-cloud,0.365192,0.300561,0.431233
4,gpt-5.4-mini,0.354073,0.279476,0.429011
5,gemini-3.1-flash-lite,0.323367,0.266230,0.378316


Ensemble minus best single model (claude-haiku-4.5):
  point estimate      +0.038
  95% interval        [-0.031, +0.102]
  P(ensemble better)  0.86


## Discrimination vs. Score Scale

Two different failure modes are hidden inside a single QWK number: whether a model *orders*
`(abstract, question)` pairs the way the expert does, and whether it places those pairs on the
0-120 axis where the fixed 40/80 cut points expect them. Spearman rho is computed on raw scores,
so it measures ordering only and is unaffected by the cut points.


In [10]:
raw_score_matrix = eval_df.pivot_table(
    index=pair_index, columns=model_col, values=prediction_col, aggfunc="first"
).dropna()
raw_score_matrix[AVERAGE_MODEL_NAME] = raw_score_matrix.mean(axis=1)
expert_raw_by_pair = (
    eval_df.drop_duplicates(pair_index)
    .set_index(pair_index)
    .loc[raw_score_matrix.index, "Evaluator_Affinity"]
)

scale_df = (
    pd.DataFrame(
        {
            "Model": raw_score_matrix.columns,
            "Spearman rho (raw score vs expert)": [
                spearmanr(raw_score_matrix[name], expert_raw_by_pair).statistic
                for name in raw_score_matrix.columns
            ],
            "Mean raw score": raw_score_matrix.mean().to_numpy(),
            "Median raw score": raw_score_matrix.median().to_numpy(),
            "Share predicted >= 40": (raw_score_matrix >= 40).mean().to_numpy(),
        }
    )
    .sort_values("Spearman rho (raw score vs expert)", ascending=False)
    .reset_index(drop=True)
)

display(scale_df)
print(f"Expert mean raw affinity: {expert_raw_by_pair.mean():.1f}")
print(f"Expert share >= 40:       {(expert_raw_by_pair >= 40).mean():.3f}")

print("\nPairwise agreement on discrete levels (fraction of pairs given the same level):")
model_slugs = [name for name in raw_score_matrix.columns if name != AVERAGE_MODEL_NAME]
display(
    pd.DataFrame(
        {a: {b: (level_matrix[a] == level_matrix[b]).mean() for b in model_slugs} for a in model_slugs}
    )
)

,Model,Spearman rho (raw score vs expert),Mean raw score,Median raw score,Share predicted >= 40
0,Average across models,0.488925,38.828904,35.0,0.443522
1,gemma4-31b-cloud,0.483420,40.431894,40.0,0.589701
2,gemini-3.1-flash-lite,0.479619,50.830565,50.0,0.687708
3,gpt-5.4-mini,0.436951,33.076412,20.0,0.342193
4,claude-haiku-4.5,0.429713,40.445183,35.0,0.471761
5,deepseek-v4-flash-cloud,0.420429,29.360465,20.0,0.373754


Expert mean raw affinity: 24.9
Expert share >= 40:       0.272

Pairwise agreement on discrete levels (fraction of pairs given the same level):


,claude-haiku-4.5,deepseek-v4-flash-cloud,gemini-3.1-flash-lite,gemma4-31b-cloud,gpt-5.4-mini
claude-haiku-4.5,1.000000,0.705980,0.589701,0.679402,0.621262
deepseek-v4-flash-cloud,0.705980,1.000000,0.543189,0.679402,0.691030
gemini-3.1-flash-lite,0.589701,0.543189,1.000000,0.750831,0.536545
gemma4-31b-cloud,0.679402,0.679402,0.750831,1.000000,0.604651
gpt-5.4-mini,0.621262,0.691030,0.536545,0.604651,1.000000


## Ensemble Aggregation: Raw Scores vs. Discretised Levels

`build_average_model_frame` averages the raw 0-120 scores first and then discretises the mean,
matching the project's `Affinity_Mean` convention. The variants below compare that choice against
averaging (or voting on) the already-discretised levels instead.



In [11]:
def discretize(scores: np.ndarray, lower_cut: float, upper_cut: float) -> np.ndarray:
    return np.where(scores < lower_cut, 0, np.where(scores < upper_cut, 1, 2))


aggregation_variants = {
    "Mean of raw scores, then discretise (current)": discretize(
        raw_score_matrix[AVERAGE_MODEL_NAME].to_numpy(), 40, 80
    ),
    "Mean of discrete levels": np.floor(level_matrix[model_slugs].mean(axis=1) + 0.5)
    .clip(0, 2)
    .astype(int)
    .to_numpy(),
    "Median of raw scores, then discretise": discretize(
        raw_score_matrix[model_slugs].median(axis=1).to_numpy(), 40, 80
    ),
    "Majority vote of discrete levels": level_matrix[model_slugs].mode(axis=1)[0].astype(int).to_numpy(),
}

aggregation_df = pd.DataFrame(
    [
        {
            "Aggregation": name,
            "Quadratic Weighted Kappa": quadratic_weighted_kappa(expert_num_by_pair, y_pred),
            "Mean signed error": float(np.mean(y_pred - expert_num_by_pair)),
            "Exact accuracy": float(np.mean(y_pred == expert_num_by_pair)),
        }
        for name, y_pred in aggregation_variants.items()
    ]
)

display(aggregation_df)

,Aggregation,Quadratic Weighted Kappa,Mean signed error,Exact accuracy
0,"Mean of raw scores, then discretise (current)",-0.019888,0.181063,0.501661
1,Mean of discrete levels,0.432701,0.275748,0.612957
2,"Median of raw scores, then discretise",-0.020082,0.279070,0.466777
3,Majority vote of discrete levels,0.421261,0.259136,0.609635


## Cut-Point Recalibration

The 40/80 boundaries in `normalize_label` are inherited from the scoring scale, not fitted to the
expert labels. Refitting them per model shows how much of each model's error is a scale offset
rather than a ranking failure. Cut points are fitted on training abstracts and scored on held-out
abstracts (grouped 5-fold, so no abstract appears on both sides), which keeps the reported gain
honest. The same protocol then tests whether dropping models from the ensemble actually helps.


In [12]:
CUT_POINT_GRID = np.arange(10, 121, 5)
CV_SPLITS = 5


def fit_cut_points(scores: np.ndarray, y_true: np.ndarray) -> tuple[int, int]:
    candidates = [
        (quadratic_weighted_kappa(y_true, discretize(scores, lower, upper)), lower, upper)
        for lower in CUT_POINT_GRID
        for upper in CUT_POINT_GRID
        if upper > lower
    ]
    _, best_lower, best_upper = max(candidates)
    return int(best_lower), int(best_upper)


abstract_groups = raw_score_matrix.index.get_level_values(0).to_numpy()
expert_num_aligned = (
    average_model_df.set_index(pair_index).loc[raw_score_matrix.index, "expert_num"].to_numpy()
)
folds = list(GroupKFold(n_splits=CV_SPLITS).split(raw_score_matrix, expert_num_aligned, abstract_groups))

recalibration_rows = []
for model_name in raw_score_matrix.columns:
    scores = raw_score_matrix[model_name].to_numpy()
    held_out_pred = np.empty(len(scores), dtype=int)
    fold_cut_points = []
    for train_rows, test_rows in folds:
        lower_cut, upper_cut = fit_cut_points(scores[train_rows], expert_num_aligned[train_rows])
        fold_cut_points.append((lower_cut, upper_cut))
        held_out_pred[test_rows] = discretize(scores[test_rows], lower_cut, upper_cut)
    recalibration_rows.append(
        {
            "Model": model_name,
            "QWK @ 40/80": quadratic_weighted_kappa(expert_num_aligned, discretize(scores, 40, 80)),
            "QWK @ refitted cuts": quadratic_weighted_kappa(expert_num_aligned, held_out_pred),
            "Mean signed error @ refitted cuts": float(np.mean(held_out_pred - expert_num_aligned)),
            "Exact accuracy @ refitted cuts": float(np.mean(held_out_pred == expert_num_aligned)),
            "Cut points (all data)": fit_cut_points(scores, expert_num_aligned),
            "Lower cut per fold": sorted({lower for lower, _ in fold_cut_points}),
            "Upper cut per fold": sorted({upper for _, upper in fold_cut_points}),
        }
    )

recalibration_df = pd.DataFrame(recalibration_rows)
recalibration_df["QWK gain"] = (
    recalibration_df["QWK @ refitted cuts"] - recalibration_df["QWK @ 40/80"]
)
recalibration_df = recalibration_df.sort_values("QWK @ refitted cuts", ascending=False).reset_index(
    drop=True
)
display(recalibration_df)

subset_scores = {
    subset: raw_score_matrix[list(subset)].mean(axis=1).to_numpy()
    for size in range(1, len(model_slugs) + 1)
    for subset in itertools.combinations(model_slugs, size)
}

subset_held_out_pred = np.empty(len(raw_score_matrix), dtype=int)
subset_picks = []
for train_rows, test_rows in folds:
    ranked = []
    for subset, scores in subset_scores.items():
        lower_cut, upper_cut = fit_cut_points(scores[train_rows], expert_num_aligned[train_rows])
        train_kappa = quadratic_weighted_kappa(
            expert_num_aligned[train_rows], discretize(scores[train_rows], lower_cut, upper_cut)
        )
        ranked.append((train_kappa, subset, lower_cut, upper_cut))
    _, subset, lower_cut, upper_cut = max(ranked, key=lambda row: row[0])
    subset_picks.append((subset, lower_cut, upper_cut))
    subset_held_out_pred[test_rows] = discretize(subset_scores[subset][test_rows], lower_cut, upper_cut)

all_models_kappa = recalibration_df.loc[
    recalibration_df["Model"] == AVERAGE_MODEL_NAME, "QWK @ refitted cuts"
].iloc[0]

print("Ensemble subset selection, scored on held-out abstracts:")
print(
    "  QWK with a subset chosen per fold: "
    f"{quadratic_weighted_kappa(expert_num_aligned, subset_held_out_pred):.3f}"
)
print(f"  QWK keeping all {len(model_slugs)} models:        {all_models_kappa:.3f}")
for subset, lower_cut, upper_cut in subset_picks:
    print(f"    fold picked {len(subset)} models at {lower_cut}/{upper_cut}: {', '.join(subset)}")

,Model,QWK @ 40/80,QWK @ refitted cuts,Mean signed error @ refitted cuts,Exact accuracy @ refitted cuts,Cut points (all data),Lower cut per fold,Upper cut per fold,QWK gain
0,gemini-3.1-flash-lite,0.323367,0.454416,0.031561,0.729236,"(70, 95)",[70],"[85, 95]",0.131050
1,Average across models,0.422301,0.434702,0.021595,0.699336,"(65, 75)","[50, 65]","[70, 75, 80]",0.012401
2,gemma4-31b-cloud,0.365192,0.416592,0.029900,0.699336,"(60, 85)","[60, 65]","[85, 90]",0.051400
3,claude-haiku-4.5,0.385572,0.410631,-0.058140,0.719269,"(65, 85)","[60, 65]","[75, 80, 85]",0.025059
4,gpt-5.4-mini,0.354073,0.379927,0.099668,0.651163,"(45, 90)","[20, 45]",[90],0.025854
5,deepseek-v4-flash-cloud,0.381463,0.370611,0.048173,0.677741,"(45, 75)","[45, 50, 55]",[75],-0.010851


Ensemble subset selection, scored on held-out abstracts:
  QWK with a subset chosen per fold: 0.470
  QWK keeping all 5 models:        0.435
    fold picked 4 models at 65/85: claude-haiku-4.5, gemini-3.1-flash-lite, gemma4-31b-cloud, gpt-5.4-mini
    fold picked 3 models at 65/75: claude-haiku-4.5, gemini-3.1-flash-lite, gemma4-31b-cloud
    fold picked 3 models at 65/75: claude-haiku-4.5, gemini-3.1-flash-lite, gemma4-31b-cloud
    fold picked 3 models at 65/75: claude-haiku-4.5, gemini-3.1-flash-lite, gemma4-31b-cloud
    fold picked 3 models at 65/85: claude-haiku-4.5, gemini-3.1-flash-lite, gemma4-31b-cloud


# Model Recommendation — Is the Cross-Model Average Better Than Isolated Models?

Cut points are methodologically defined (Low `< 40`, Moderate `40-80`, High `>= 80`) and are treated
as fixed throughout. Everything below is measured at those boundaries; no recalibration is proposed.

Basis: benchmark run `20260726_134607_4` — 5 models, scored independently against two expert sets,
`20260728_abstracts_evaluation_template_mark_om_v1` (Mark, 601 pairs / 91 abstracts) and
`20260728_abstracts_evaluation_template_charlie_s_v1` (Charlie, 720 pairs / 99 abstracts). The cells
above load Mark only; the Charlie column and the paired cluster bootstrap were produced by the same
logic run against the second expert set, and are reported here for robustness.

## Answer

**Yes against the average single model — no against the best one.** The cross-model average is never
significantly worse than any individual model against either expert, and against Mark it beats all
five. But it does not beat `deepseek-v4-flash-cloud`, which ties it on QWK against Charlie and leads
on accuracy and MAE against both experts.

Use the ensemble as the default. Its value is **insurance rather than a performance gain**: you do
not need to know in advance which model is strongest, and it neutralises `gemini-3.1-flash-lite`'s
failure mode (QWK 0.324 / 0.209) completely.

## Expert set A — Mark (601 pairs, 91 abstracts, expert mix 437/137/27)

| Model | QWK | Accuracy | MAE | Bias | Balanced acc. | Recall High |
|---|---|---|---|---|---|---|
| **ENSEMBLE (avg across models)** | **0.450** | 0.644 | 0.378 | +0.225 | **0.596** | **0.519** |
| claude-haiku-4.5 | 0.387 | 0.607 | 0.403 | +0.200 | 0.483 | 0.222 |
| deepseek-v4-flash-cloud | 0.383 | **0.682** | **0.354** | **+0.118** | 0.534 | 0.296 |
| mistral-small-2603 | 0.354 | 0.562 | 0.486 | +0.290 | 0.530 | 0.593 |
| gpt-5.4-mini | 0.354 | 0.637 | 0.421 | +0.151 | 0.467 | 0.333 |
| gemini-3.1-flash-lite | 0.324 | 0.444 | 0.606 | +0.532 | 0.536 | 0.667 |

Paired cluster bootstrap (resampling abstracts, 2000 draws, seed 0), ensemble QWK minus single-model
QWK — **all five intervals lie above zero**:

| vs. single model | Δ QWK | 95% interval | P(ensemble better) |
|---|---|---|---|
| gemini-3.1-flash-lite | +0.126 | [+0.075, +0.177] | 1.00 |
| gpt-5.4-mini | +0.096 | [+0.041, +0.153] | 1.00 |
| mistral-small-2603 | +0.095 | [+0.052, +0.141] | 1.00 |
| deepseek-v4-flash-cloud | +0.067 | [+0.016, +0.120] | 0.99 |
| claude-haiku-4.5 | +0.065 | [+0.008, +0.122] | 0.99 |

## Expert set B — Charlie (720 pairs, 99 abstracts, expert mix 616/104/0)

| Model | QWK | Accuracy | MAE | Bias | Balanced acc. | Recall Moderate |
|---|---|---|---|---|---|---|
| deepseek-v4-flash-cloud | **0.351** | **0.797** | **0.221** | **+0.096** | 0.638 | 0.413 |
| **ENSEMBLE (avg across models)** | 0.347 | 0.740 | 0.279 | +0.193 | 0.632 | 0.481 |
| claude-haiku-4.5 | 0.317 | 0.725 | 0.286 | +0.197 | **0.651** | **0.548** |
| gpt-5.4-mini | 0.306 | 0.757 | 0.275 | +0.156 | 0.590 | 0.356 |
| mistral-small-2603 | 0.275 | 0.692 | 0.353 | +0.256 | 0.548 | 0.346 |
| gemini-3.1-flash-lite | 0.209 | 0.528 | 0.525 | +0.489 | 0.532 | 0.538 |

| vs. single model | Δ QWK | 95% interval | Verdict |
|---|---|---|---|
| gemini-3.1-flash-lite | +0.138 | [+0.082, +0.196] | ensemble wins |
| mistral-small-2603 | +0.072 | [+0.026, +0.119] | ensemble wins |
| gpt-5.4-mini | +0.042 | [−0.017, +0.104] | not significant |
| claude-haiku-4.5 | +0.031 | [−0.017, +0.079] | not significant |
| deepseek-v4-flash-cloud | −0.003 | [−0.061, +0.061] | tied |

Charlie assigned **no** High labels, so High recall is undefined for every model in this column and
the ensemble's main advantage over `deepseek` in set A cannot be tested here.

## Why the answer depends on the metric

The ensemble leads on **QWK, balanced accuracy, and minority-class recall**. `deepseek-v4-flash-cloud`
leads on **plain accuracy, MAE, and bias** against both experts.

The mechanism is that the ensemble spreads its predictions across classes more, which recovers
minority-class cases — High recall 0.519 vs. `deepseek`'s 0.296 against Mark — and that is exactly
what QWK's chance correction and balanced accuracy reward. It pays for this with extra errors on the
dominant Low class, which is what plain accuracy and MAE penalise. Pair-for-pair, `deepseek` is
actually closer to the expert more often than the ensemble is:

| Ensemble vs. deepseek, per pair | Ensemble closer | Tie | Ensemble worse |
|---|---|---|---|
| Mark | 8.3% | 81.0% | 10.6% |
| Charlie | 3.6% | 86.9% | 9.4% |

So: if the deciding metric is ordinal agreement with the expert, keep the ensemble. If it is
exact-bucket accuracy or unbiasedness, `deepseek-v4-flash-cloud` alone is as good or better at one
fifth of the calls.

## Single-model rankings are unstable across experts

`mistral-small-2603` places 3rd against Mark and 5th against Charlie; `claude-haiku-4.5` 2nd and 3rd;
`gpt-5.4-mini` 4th and 4th. Only `deepseek-v4-flash-cloud` and the ensemble place 1st or 2nd against
both. This instability is the strongest practical argument for averaging: picking a single model on
one expert's labels risks picking a model that ranks mid-pack on another's.

## Caveats

- 91-99 abstracts per expert set, one annotator each. Single-model QWK differences of less than about
  0.05 are not resolvable at this sample size.
- The ensemble's clearest advantage — minority-class recall — rests on Mark's 27 High pairs, and
  Charlie's set has none.
- Every model over-predicts (bias +0.10 to +0.53 across both experts). With cut points fixed, this is
  a property to report, not one to tune away.
